# 07 Caching

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsc-wdc/qdislib/blob/main/notebooks/07_caching.ipynb) [![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/bsc-wdc/qdislib/main?labpath=notebooks/07_caching.ipynb)

A cut runs many fragment circuits, and much of that work comes back: the terms of a Hamiltonian share fragments, a variational loop changes only some fragments per step,
and runs get repeated. The **cache** keeps what each fragment circuit returned, its counts or its exact correlators, so a circuit already run is read back. One entry serves every observable
later read from that circuit.

In [1]:
# Installs Qdislib on Colab or Binder; does nothing where Qdislib 2 is installed.
try:
    import Qdislib
    assert Qdislib.__version__.startswith("2.")
except (ImportError, AssertionError):
    %pip install -q "qdislib[qiskit,cache]>=2.0"

In [2]:
from qiskit import QuantumCircuit
import Qdislib as qd


def example_circuit():
    """Five qubits in a line, joined by four CZ gates."""
    qc = QuantumCircuit(5)
    for qubit, angle in enumerate([0.8, 0.5, 0.6, 0.4, 0.3]):
        qc.ry(angle, qubit)
    for qubit in range(4):
        qc.cz(qubit, qubit + 1)
    for qubit, angle in enumerate([0.2, 0.1, 0.7, 0.3, 0.9]):
        qc.rx(angle, qubit)
    return qc


circuit = example_circuit()
cut = ["CZ_7"]


hamiltonian = [(0.5, "ZZZZZ"), (1.0, "XIIII"), (-0.3, "IIIIZ")]

## Per call: `cache=`

`cache=` on every cutting function takes `True`, `"memory"`, `"lmdb:<path>"`, `"redis://host:port/db"` or a store object. A store object lets you look at its statistics.
`size` is how many results are stored. The first call below runs 15 different circuits, not 24, because the two commuting groups of the Hamiltonian share the fragments their basis does not change. The repeated call runs none, and returns the same value.

In [3]:
store = qd.default_circuit_cache("memory")

first = qd.circuit_cutting(circuit, cut, hamiltonian, shots=4000, seed=1, cache=store)
print("first: ", round(first, 4), store.stats())

second = qd.circuit_cutting(circuit, cut, hamiltonian, shots=4000, seed=1, cache=store)
print("second:", round(second, 4), store.stats())

print("subcircuits without the cache:", len(qd.circuit_cutting_subcircuits(circuit, cut, hamiltonian)))

first:  0.5701 {'hits': 0, 'misses': 30, 'hit_rate': 0.0, 'size': 15}
second: 0.5701 {'hits': 24, 'misses': 30, 'hit_rate': 0.4444444444444444, 'size': 15}
subcircuits without the cache: 24


Another observable in a basis that has run needs nothing new. A different basis runs only the fragments it changes.

In [4]:
before = store.stats()["size"]
value = qd.circuit_cutting(circuit, cut, "ZZIII", shots=4000, seed=1, cache=store)
print("ZZIII: new entries", store.stats()["size"] - before, " value", round(value, 4))

before = store.stats()["size"]
value = qd.circuit_cutting(circuit, cut, "IIIIX", shots=4000, seed=1, cache=store)
print("IIIIX: new entries", store.stats()["size"] - before, " value", round(value, 4))

ZZIII: new entries 0  value 0.5992
IIIIX: new entries 5  value 0.2692


## Shots are reused too

A shot result is reused by later runs: the key holds the backend and the shots, not the seed, so a repeated run returns the same samples and costs no device time.
This is what you want for a repeated workflow and not what you want to measure spread. `cache_refresh=True` runs every circuit again and replaces what is stored.

In [5]:
store = qd.default_circuit_cache("memory")
a = qd.gate_cutting(circuit, cut, shots=2000, seed=1, cache=store)
b = qd.gate_cutting(circuit, cut, shots=2000, seed=2, cache=store)
c = qd.gate_cutting(circuit, cut, shots=2000, seed=2, cache=store, cache_refresh=True)
print(f"seed=1 {a:.4f}   seed=2 from the cache {b:.4f}   seed=2 refreshed {c:.4f}")

seed=1 0.2527   seed=2 from the cache 0.2527   seed=2 refreshed 0.2364


## For a session: `use_cache`

`qd.use_cache(...)` applies to every later call, also as a context manager. `cache=False` turns it off for one call.

In [6]:
with qd.use_cache(True):
    first = qd.gate_cutting(circuit, cut, shots=2000)
    second = qd.gate_cutting(circuit, cut, shots=2000)
    third = qd.gate_cutting(circuit, cut, shots=2000, cache=False)
print(f"{first:.4f} {second:.4f} (read back)   {third:.4f} (ran)")

0.2148 0.2148 (read back)   0.2363 (ran)


## A variational loop

When one parameter moves, the fragments it does not touch come back unchanged. The angle below is only on qubit 4, in the second fragment, so the first fragment's circuits run once.

In [7]:
def ansatz(theta):
    qc = example_circuit()
    qc.ry(theta, 4)
    return qc


store = qd.default_circuit_cache("memory")
angles = [0.1, 0.2, 0.3, 0.4, 0.5]
for theta in angles:
    qd.gate_cutting(ansatz(theta), cut, shots=2000, seed=1, cache=store)
    print(f"theta {theta}: stored so far {store.stats()['size']}")
print("subcircuits per step:", len(qd.gate_cutting_subcircuits(ansatz(0.1), cut)), "; without the cache, in total:",
      len(angles) * len(qd.gate_cutting_subcircuits(ansatz(0.1), cut)))

theta 0.1: stored so far 10


theta 0.2: stored so far 15
theta 0.3: stored so far 20
theta 0.4: stored so far 25
theta 0.5: stored so far 30
subcircuits per step: 12 ; without the cache, in total: 60


## What counts as the same circuit: `cache_key`

- `"exact"` (the default): the same fragment circuit on the same backend with the same shots. It is computed on the master without building the circuit, so a hit never becomes a task and never matches another circuit.
- `"semantic"`: also equivalent circuits, by a hash of the ZX-reduced graph, computed in the task (`pip install "qdislib[cache]"` for PyZX). ZX reduction is not a canonical form, so an equivalent circuit can still be missed. It is then run, not served wrongly.

`qd.semantic_key` is that hash. Three Hadamards reduce to one:

In [8]:
one = QuantumCircuit(2)
one.h(0)
one.cz(0, 1)

three = QuantumCircuit(2)
three.h(0)
three.h(0)
three.h(0)
three.cz(0, 1)

other = QuantumCircuit(2)
other.h(1)
other.cz(0, 1)

print("H vs H H H:       ", qd.semantic_key(one) == qd.semantic_key(three))
print("H on another qubit:", qd.semantic_key(one) == qd.semantic_key(other))

store = qd.default_circuit_cache("memory")
qd.gate_cutting(circuit, cut, shots=2000, seed=1, cache=store, cache_key="semantic")
print(store.stats())

H vs H H H:        True
H on another qubit: False


{'hits': 0, 'misses': 10, 'hit_rate': 0.0, 'size': 10}


## Persistent stores

In memory, `max_entries` evicts the least recently used entry and `ttl` expires entries after some seconds, useful after an SDK upgrade. On disk, LMDB outlives the process. Under PyCOMPSs `cache=True`
is an LMDB store in the working directory (or `$QDISLIB_CACHE_DIR`), shared by the workers. Fragments wider than 18 qubits are not stored exactly, as their correlator vector would be too large.

In [9]:
import tempfile

with tempfile.TemporaryDirectory() as directory:
    spec = "lmdb:" + directory + "/cache"
    first = qd.gate_cutting(circuit, cut, shots=2000, seed=1, cache=spec)
    second = qd.gate_cutting(circuit, cut, shots=2000, seed=2, cache=spec)
    print(first == second)

bounded = qd.default_circuit_cache("memory", max_entries=5000, ttl=3600)
print(qd.gate_cutting(circuit, cut, shots=2000, seed=1, cache=bounded), bounded.stats())

True


0.2527025 {'hits': 0, 'misses': 20, 'hit_rate': 0.0, 'size': 10}


Next: **08 Backend internals**.